In [ ]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

from anngeno import AnnGeno
from scripts import get_burdens
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Get missense variants for required gene

In [ ]:
pg = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').filter(pl.col('gene_name').is_in(['BRCA1']))
pg

In [ ]:
pg['file_name'].value_counts().sort('count', descending=True)

In [ ]:
plt.hist(pg['dms_score'], bins=100)
plt.show()

In [ ]:
anno = pl.read_parquet('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag/annotations.parquet').with_columns(
    pl.when(
        pl.col('amino_acids').is_not_null() & pl.col('protein_position').is_not_null()
    ).then(
        pl.col('amino_acids').str.split('/').list.get(0) +
        pl.col('protein_position').str.split('/').list.get(0) +
        pl.col('amino_acids').str.split('/').list.get(1)
    ).otherwise(None).alias('mutant')
).filter(
    # Filter for BRCA1 gene
    pl.col('region').is_in(['ENSG00000012048'])
)

id_cols = ['chrom', 'pos', 'ref', 'alt', 'id', 'region', 'col', 'AF_ukb', 'mutant', 'amino_acids', 'protein_position', 'consequence']
missense_annos = ['loftee_hc', 'CADD_RAW', 'am_pathogenicity', 'Consequence_missense_variant', 'PolyPhen', 'CADD_SIFTval', 'CADD_priPhCons', 'CADD_mamPhCons', 'CADD_verPhCons', 'gpn_score']

anno = anno.select(id_cols + missense_annos)
anno

In [ ]:
anno.filter(pl.col('Consequence_missense_variant')==1)

In [ ]:
pg.filter(~pl.col('mutant').is_in(anno['mutant']))

In [ ]:
brca_df = pg.filter(pl.col('file_name').is_in(['BRCA1_HUMAN_Findlay_2018']))[['mutant', 'dms_score', 'gene_name', 'file_name', 'region']].join(anno.filter(pl.col('region') == 'ENSG00000012048'), on=['region', 'mutant'], how='inner')

annos2compare = missense_annos + ['dms_score']
brca_df

### Check how exp. scores correlate with comp. scores

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='am_pathogenicity')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='Alphamissense') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='CADD_RAW')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='CADD RAW') +
    theme_bw()
)

In [ ]:
(
    ggplot(brca_df, aes(x='dms_score', y='gpn_score')) +
    geom_point() +
    geom_smooth(method='lm', se=True, color='darkred') +
    labs(x='DMS Score', y='GPN-MSA') +
    theme_bw()
)

### Pos-neg split scores

In [ ]:

def check_positive_negative(df: pl.DataFrame, columns):
    results = {}
    for col in columns:
        if col in df.columns:
            non_null = df.select(pl.col(col).drop_nulls())[col]
            if non_null.is_empty():
                results[col] = False  # Only nulls
            else:
                min_val = non_null.min()
                max_val = non_null.max()
                results[col] = (min_val < 0) and (max_val > 0)
        else:
            results[col] = False  # Column not found
    return results

# Example usage:
positive_negative_check = check_positive_negative(brca_df, annos2compare)

# Print the results
for column, has_both in positive_negative_check.items():
    if has_both:
        print(f"Column '{column}': Contains both positive and negative values.")

In [ ]:
def split_pos_neg_lazy(df: pl.LazyFrame, columns):
    # Start with the lazy frame
    lf = df

    for col in columns:
        if col in df.columns:
            pos_col = (
                pl.when(pl.col(col) > 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_pos")
            )

            neg_col = (
                pl.when(pl.col(col) < 0)
                .then(pl.col(col))
                .otherwise(0)
                .alias(f"{col}_neg")
            )

            lf = lf.with_columns([pos_col, neg_col])

    return lf

# Get the columns that contain both positive and negative values
mix_cols = [k for k, v in positive_negative_check.items() if v]

# Split the positive and negative values into separate columns
split_ann = split_pos_neg_lazy(brca_df.lazy(), mix_cols).collect()
split_ann

## Get genotypes and delta phenotype

### Get phenotypes and PRS

In [ ]:
phenotype = 'jurgens_breast_cancer'

phenos = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/phenotypes190_missing80_unique2.parquet').select(['individual', phenotype]).drop_nulls()

prs = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/PRS190_missing80_unique2.parquet').select(['individual', f'{phenotype}_prs']).drop_nulls()

In [ ]:
config_path = f'/home/dnanexus/ukbgym/config_dms.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get("covariates")

cov_df = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet').rename({'eid':'individual'}).select(['individual'] + cov_list).with_columns(pl.col('individual').cast(pl.Int64).alias('individual'))
cov_df

all_df = phenos.join(prs, on='individual', how='inner').join(cov_df, on='individual', how='inner')
all_pd = all_df.to_pandas()

In [ ]:
# Restrict to EUR ancestry
eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv')
eur_samples

all_pd = all_pd[all_pd['individual'].isin(eur_samples['eid'].to_list())]
all_pd

In [ ]:
import statsmodels.api as sm

combined_df = pd.DataFrame(index=all_pd.index)

y = all_pd[phenotype]
X = all_pd.drop(columns=[phenotype])
X = sm.add_constant(X)  # Add a constant term for the intercept

# Fit the model
model = sm.OLS(y, X).fit()

# Save residuals
residuals = pd.Series(model.resid, index=combined_df.index, name=f'{phenotype}_residual')

pdf = pl.DataFrame(pd.concat([all_pd[['individual', phenotype, f'{phenotype}_prs']], residuals], axis=1)).with_columns(pl.col('individual').cast(pl.String).alias('individual'))
pdf

### Get genotypes

In [ ]:
with open(config_path) as f:
    config = yaml.safe_load(f)

maf = config.get('maf', None)

In [ ]:
ag = AnnGeno('/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag', mode='r', low_mem=True)
ag._set_annotations(split_ann.lazy())

variants_to_keep = ag.annotations.filter((pl.col('AF_ukb') < maf)).select('id').collect()['id']
ag.subset_variants(set(variants_to_keep))

samples = ag.samples
samples

In [ ]:
gene_id = 'ENSG00000012048'
reg_dict = ag.get_region(gene_id)

geno = reg_dict['genotypes']
anno_df = reg_dict['annotations']
geno

In [ ]:
# Find where the genotype is 1
rows, cols = np.where(geno == 1)

# Build the melted DataFrame
het = pl.DataFrame({
    'id': np.array(anno_df['id'])[rows],
    'individual': np.array(samples)[cols],
    'genotype': 1  # since we filtered for 1s only
})

# Homozygous genotypes
rows, cols = np.where(geno == 2)
hom = pl.DataFrame({
    'id': np.array(anno_df['id'])[rows],
    'individual': np.array(samples)[cols],
    'genotype': 2  # since we filtered for 1s only
})

geno_melt = pl.concat([het, hom])
geno_melt

In [ ]:
geno_melt['id'].value_counts().sort('count', descending=True)

In [ ]:
# Will be subset to the EUR samples if pdf is subset
pheno_melt = geno_melt.join(pdf, on='individual', how='inner')
pheno_melt = pheno_melt.filter(
    pl.col('genotype')==1
).group_by(['id']).agg(
    pl.col(f'{phenotype}').mean().alias(f'average_{phenotype}'),
    pl.col(f'{phenotype}_residual').mean().alias(f'average_{phenotype}_residual')
)

pheno_melt

In [ ]:
all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)
all_annotation_list = list(set(all_annotation_list).intersection(set(ag.annotations.collect_schema().names())))

anno_melt = anno_df.unpivot(
    index=['chrom', 'pos', 'ref', 'alt', 'id', 'region', 'col', 'AF_ukb', 'mutant', 'amino_acids', 'protein_position'],
    on=all_annotation_list,
    variable_name='annotation',
    value_name='score'
)
anno_melt

In [ ]:
plt_df = pheno_melt.join(anno_melt, on='id', how='inner')
plt_df

## Spearman (rank) Correlation plots

In [ ]:
from scipy.stats import spearmanr

anno = 'am_pathogenicity'
pheno_col = f'average_{phenotype}_residual' 

# Add ranks manually
df_filtered = plt_df.filter(
        pl.col('annotation') == anno
    ).with_columns([
        pl.col('score').rank().alias('score_rank'),
        pl.col(pheno_col).rank().alias('pheno_rank')
    ])

corr, pval = spearmanr(df_filtered['score_rank'], df_filtered['pheno_rank'])
corr_text = f'Spearman ρ = {corr:.2f}, p = {pval:.3g}'

(
    # ggplot(df_filtered, aes(x='score_rank', y='pheno_rank')) +
    ggplot(df_filtered, aes(x='score', y=pheno_col)) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='darkred') +
    theme_classic() +
    labs(x='Alphamissense', y=f'Average breast cancer risk (residual)') +
    annotate(
        'text',
        x=df_filtered['score'].min(),
        y=df_filtered[pheno_col].max(),
        label=corr_text,
        ha='left',
        va='top',
        size=12
    ) +
    theme(figure_size=(5, 4))
)


In [ ]:
from scipy.stats import spearmanr

anno = 'dms_score'
pheno_col = f'average_{phenotype}_residual'

# Add ranks manually
df_filtered = plt_df.filter(
        pl.col('annotation') == anno
    ).with_columns([
        pl.col('score').rank().alias('score_rank'),
        pl.col(pheno_col).rank().alias('pheno_rank')
    ])

corr, pval = spearmanr(df_filtered['score_rank'], df_filtered['pheno_rank'])
corr_text = f'Spearman ρ = {corr:.2f}, p = {pval:.3g}'

(
    # ggplot(df_filtered, aes(x='score_rank', y='pheno_rank')) +
    ggplot(df_filtered, aes(x='score', y=pheno_col)) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='darkred') +
    theme_classic() +
    labs(x='DMS score', y=f'Average breast cancer risk (residual)') +
    annotate(
        'text',
        x=df_filtered['score'].min(),
        y=df_filtered[pheno_col].max(),
        label=corr_text,
        ha='left',
        va='top',
        size=12
    ) +
    theme(figure_size=(5, 4))
)


In [ ]:
from scipy.stats import spearmanr

pheno_col = pheno_col = f'average_{phenotype}_residual'
corr_results = []

# Group by annotation and calculate correlation
for annotation, group in plt_df.group_by('annotation'):
    # Drop NA in the relevant columns
    clean_group = group.drop_nans(subset=['score', pheno_col])
    if len(clean_group) > 1:  # Need at least 2 points to calculate correlation
        corr, pval = spearmanr(clean_group['score'], clean_group[pheno_col])
        corr_results.append({'annotation': annotation[0], 'abs_correlation': np.abs(corr), 'p_value': pval})
    else:
        corr_results.append({'annotation': annotation[0], 'abs_correlation': None, 'p_value': None})

# Convert to DataFrame
rank_corr_df = pd.DataFrame(corr_results).dropna()
rank_corr_df

In [ ]:
drop_anno = [a for a in rank_corr_df['annotation'] if a.endswith('_pos') or a.endswith('_neg')]

rank_corr_df = rank_corr_df[~rank_corr_df['annotation'].isin(drop_anno)].sort_values('abs_correlation', ascending=False)
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)

rank_corr_df['color_dms'] = rank_corr_df['annotation'].str.startswith('dms_').map({True: 'DMS', False: 'Other'})

# Plot bar plot of correlations
(
    ggplot(rank_corr_df, aes(x='annotation', y='abs_correlation', fill='color_dms')) +
    geom_col(alpha=0.8) +
    coord_flip() +  # Flip for better readability if many annotations
    theme_bw() +
    labs(
        x='Annotation',
        y='absolute Spearman correlation'
    ) +
    theme(
        figure_size=(8, 3),
        legend_position='none'
    )
)
